# 🎭 Deep-Live-Cam — Colab WebRTC Streaming Server

Real-time face swap streaming: **Local Webcam → WebRTC → Colab GPU → Deep-Live-Cam → WebRTC → Local PC**

Run cells **1 → 8** sequentially. Cell **9** provides a monitoring dashboard.

| Cell | Purpose |
|------|----------|
| 1 | Install dependencies & verify GPU |
| 2 | Upload source face image |
| 3 | Configure Deep-Live-Cam parameters |
| 4 | Configure Ngrok tunnel |
| 5 | Launch signaling server + initialize DLC |
| 6 | Start processing pipeline |
| 7 | Generate local client.py |
| 8 | Monitoring dashboard |
| 9 | Troubleshooting reference |

In [ ]:
#@title 1️⃣ Install Dependencies & Verify GPU
#@markdown Installs all required packages, clones Deep-Live-Cam, and verifies GPU availability.

import subprocess, sys, os

# Check numpy version and downgrade if needed
try:
    import numpy
    if int(numpy.__version__.split(".")[0]) >= 2:
        print("Numpy 2.x detected. Downgrading to 1.26.4 to avoid compatibility issues...")
        subprocess.run([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "scipy==1.13.1", "-q"], check=True)
        print("\n============================================================")
        print("  🔄 RUNTIME RESTART REQUIRED")
        print("============================================================")
        print("The runtime is restarting to apply the numpy downgrade.")
        print("Please wait 5 seconds, then RUN THIS CELL AGAIN.")
        import time; time.sleep(2)
        os.kill(os.getpid(), 9)
except Exception:
    pass

print("=" * 60)
print("  STEP 1: Installing Dependencies")
print("=" * 60)

# System packages
subprocess.run(["apt-get", "update", "-qq"], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "-qq",
                "ffmpeg", "libgl1-mesa-glx", "libglib2.0-0"],
               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("[✓] System packages installed (ffmpeg, libgl1)")

# Clone Deep-Live-Cam
DLC_DIR = "/content/Deep-Live-Cam"
if not os.path.exists(DLC_DIR):
    subprocess.run(["git", "clone",
                    "https://github.com/hacksider/Deep-Live-Cam.git",
                    DLC_DIR], check=True)
    print(f"[✓] Deep-Live-Cam cloned to {DLC_DIR}")
else:
    print(f"[✓] Deep-Live-Cam already exists at {DLC_DIR}")

# Install DLC requirements (skip GUI / platform-specific deps)
req_path = os.path.join(DLC_DIR, "requirements.txt")
with open(req_path, "r") as f:
    lines = f.readlines()
filtered = []
SKIP = ["pyside6", "pygrabber", "cv2_enumerate",
        "onnxruntime-silicon", "tensorflow", "opennsfw2", "onnxruntime-gpu"]
for line in lines:
    s = line.strip()
    if not s or s.startswith("#"):
        continue
    if any(skip in s.lower() for skip in SKIP):
        continue
    filtered.append(s)
if filtered:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + filtered, check=True)
    print(f"[✓] DLC core dependencies installed ({len(filtered)} packages)")

# WebRTC + server packages
webrtc_deps = [
    "aiortc>=1.9.0", "aiohttp>=3.9.0", "av>=12.0.0",
    "pyngrok>=7.0.0", "opencv-python>=4.8.0", "onnxruntime-gpu==1.21.0",
    "numpy==1.26.4", "scipy==1.13.1",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + webrtc_deps, check=True)
print("[✓] WebRTC / server dependencies installed")

# InsightFace
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface==0.7.3"], check=True)
print("[✓] InsightFace installed")

print()
print("=" * 60)
print("  STEP 2: GPU Verification")
print("=" * 60)

try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"[✓] CUDA GPU: {gpu_name}")
        print(f"    VRAM: {gpu_mem:.1f} GB")
        print(f"    CUDA: {torch.version.cuda}  |  PyTorch: {torch.__version__}")
    else:
        print("[✗] No CUDA GPU! Runtime → Change runtime type → GPU")
except ImportError:
    print("[!] PyTorch unavailable — GPU blending will use CPU fallback")

import onnxruntime
ort_providers = onnxruntime.get_available_providers()
print(f"\n[✓] ONNX Runtime {onnxruntime.__version__}")
print(f"    Providers: {ort_providers}")
if "CUDAExecutionProvider" in ort_providers:
    print("    [✓] CUDAExecutionProvider available")
else:
    print("    [✗] CUDAExecutionProvider NOT available — inference will be slow")

import shutil
if shutil.which("ffmpeg"):
    print(f"\n[✓] ffmpeg found at {shutil.which('ffmpeg')}")
else:
    print("\n[✗] ffmpeg not found!")

print("\n" + "=" * 60)
print("  ✅ Environment ready!")
print("=" * 60)

## 2️⃣ Upload Source Face Image
Upload a clear, front-facing photo of the face you want to swap onto the webcam stream.

In [ ]:
#@title Upload Source Face Image
#@markdown Upload a clear front-facing photo (PNG / JPG).

import os, sys, cv2
import numpy as np
from IPython.display import display, Image as IPImage
from google.colab import files

DLC_DIR = "/content/Deep-Live-Cam"
SOURCE_PATH = os.path.join(DLC_DIR, "source_face.png")

print("📸 Upload a source face image (PNG/JPG):")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]
    with open(SOURCE_PATH, "wb") as f:
        f.write(file_bytes)
    print(f"[✓] Saved to {SOURCE_PATH}")
    display(IPImage(data=file_bytes, width=300))

    # Validate face detection
    sys.path.insert(0, DLC_DIR)
    import modules.globals
    modules.globals.execution_providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    from modules.face_analyser import get_one_face
    img = cv2.imread(SOURCE_PATH)
    face = get_one_face(img)
    if face is not None:
        print(f"[✓] Face detected (confidence: {face.det_score:.3f})")
    else:
        print("[✗] No face detected — upload a clearer image.")
else:
    print("[!] No file uploaded. Run this cell again.")

## 3️⃣ Configure Deep-Live-Cam
Edit the parameters below to control the face-swap pipeline.

In [ ]:
#@title Deep-Live-Cam Configuration
#@markdown Adjust parameters to control quality, speed, and features.

# === Frame Processors ===
frame_processors = ["face_swapper"]  #@param {type:"raw"}
#@markdown Available: `face_swapper`, `face_enhancer`, `face_enhancer_gpen256`, `face_enhancer_gpen512`

# === Face Detection ===
many_faces = False  #@param {type:"boolean"}
face_detection_confidence = 0.5  #@param {type:"slider", min:0.1, max:1.0, step:0.05}

# === Face Swap Options ===
opacity = 1.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
sharpness = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.1}
mouth_mask = False  #@param {type:"boolean"}
poisson_blend = False  #@param {type:"boolean"}
color_correction = False  #@param {type:"boolean"}

# === Performance ===
processing_resolution = 640  #@param {type:"slider", min:320, max:1280, step:64}
fps_limit = 30  #@param {type:"slider", min:10, max:60, step:5}
execution_threads = 2  #@param {type:"slider", min:1, max:8, step:1}
max_memory_gb = 8  #@param {type:"slider", min:4, max:16, step:2}

# === Temporal Smoothing ===
enable_interpolation = False  #@param {type:"boolean"}
interpolation_weight = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.1}

# Store config
CONFIG = dict(
    frame_processors=frame_processors, many_faces=many_faces,
    face_detection_confidence=face_detection_confidence,
    opacity=opacity, sharpness=sharpness, mouth_mask=mouth_mask,
    poisson_blend=poisson_blend, color_correction=color_correction,
    processing_resolution=processing_resolution, fps_limit=fps_limit,
    execution_threads=execution_threads, max_memory_gb=max_memory_gb,
    enable_interpolation=enable_interpolation,
    interpolation_weight=interpolation_weight,
)

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print("\n[✓] Config ready. Proceed to Cell 4.")

## 4️⃣ Configure Ngrok
Enter your Ngrok auth token to create a public HTTPS tunnel.  
Get a free token at [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
#@title Ngrok Authentication
#@markdown Your token is entered securely and never stored.

import os, time, threading
from getpass import getpass
from pyngrok import ngrok, conf

NGROK_AUTH_TOKEN = getpass("Enter your NGROK_AUTH_TOKEN: ")
if not NGROK_AUTH_TOKEN or len(NGROK_AUTH_TOKEN) < 10:
    raise ValueError("Invalid token — get one at https://dashboard.ngrok.com")

conf.get_default().auth_token = NGROK_AUTH_TOKEN
conf.get_default().region = "us"
print("[✓] Ngrok authenticated")

import socket
def get_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        return s.getsockname()[1]

SERVER_PORT = get_free_port()
_ngrok_tunnel = None
_ngrok_url = None
_ngrok_monitor_running = False

def start_ngrok_tunnel():
    global _ngrok_tunnel, _ngrok_url
    try:
        ngrok.kill()
        time.sleep(1)
        _ngrok_tunnel = ngrok.connect(SERVER_PORT, "http", bind_tls=True)
        _ngrok_url = _ngrok_tunnel.public_url
        sep = "=" * 60
        print(f"\n{sep}")
        print(f"  🌐 NGROK TUNNEL ACTIVE")
        print(f"{sep}")
        print(f"  URL: {_ngrok_url}")
        print(f"  Health: {_ngrok_url}/health")
        print(f"{sep}")
        return _ngrok_url
    except Exception as e:
        print(f"[✗] Ngrok error: {e}")
        return None

def _ngrok_monitor():
    global _ngrok_monitor_running
    _ngrok_monitor_running = True
    while _ngrok_monitor_running:
        time.sleep(30)
        try:
            if not ngrok.get_tunnels():
                print("\n[!] Ngrok tunnel dropped — reconnecting...")
                start_ngrok_tunnel()
        except Exception:
            try:
                start_ngrok_tunnel()
            except Exception:
                pass

url = start_ngrok_tunnel()
if url:
    threading.Thread(target=_ngrok_monitor, daemon=True).start()
    print("\n[✓] Auto-reconnect monitor active")
    print("[✓] Ngrok ready. Proceed to Cell 5.")
else:
    print("[✗] Failed — check your auth token.")

## 5️⃣ Initialize Pipeline & Launch Server
This cell loads the DLC models, initializes the face swap pipeline with CUDA,
and starts the WebRTC signaling server.

In [ ]:
#@title Initialize DLC & Launch Signaling Server
#@markdown Loads models, warms up GPU, and starts the WebSocket server.

import os, sys, time, asyncio, threading, logging
# ── WebRTC Imports (Must be BEFORE cv2/torch to prevent FFmpeg segfaults) ──
import av
from av import VideoFrame
from aiohttp import web
from aiortc import (
    RTCPeerConnection, RTCSessionDescription,
    MediaStreamTrack, RTCConfiguration, RTCIceServer,
)
from pyngrok import ngrok, conf
import cv2
import numpy as np

# ── DLC Initialization ──────────────────────────────────────
DLC_DIR = "/content/Deep-Live-Cam"
sys.path.insert(0, DLC_DIR)
os.chdir(DLC_DIR)

os.environ["OMP_NUM_THREADS"] = "6"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Patch core.py to remove ui import so we do not need PySide6 in headless mode
core_path = os.path.join(DLC_DIR, "modules", "core.py")
with open(core_path, "r", encoding="utf-8") as f:
    core_code = f.read()
core_code = core_code.replace("import modules.ui as ui", "# import modules.ui as ui")
with open(core_path, "w", encoding="utf-8") as f:
    f.write(core_code)

# Patch face_swapper.py to disable CUDA graphs (causes segfaults in Colab)
swapper_path = os.path.join(DLC_DIR, "modules", "processors", "frame", "face_swapper.py")
with open(swapper_path, "r", encoding="utf-8") as f:
    swapper_code = f.read()
swapper_code = swapper_code.replace("_init_cuda_graph_session(model_path, FACE_SWAPPER)", "pass  # Disabled CUDA graphs")
with open(swapper_path, "w", encoding="utf-8") as f:
    f.write(swapper_code)

# Set globals
import modules.globals
modules.globals.execution_providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
modules.globals.execution_threads = CONFIG.get("execution_threads", 2)
modules.globals.max_memory = CONFIG.get("max_memory_gb", 8)
modules.globals.frame_processors = CONFIG.get("frame_processors", ["face_swapper"])
modules.globals.many_faces = CONFIG.get("many_faces", False)
modules.globals.map_faces = False
modules.globals.mouth_mask = CONFIG.get("mouth_mask", False)
modules.globals.poisson_blend = CONFIG.get("poisson_blend", False)
modules.globals.color_correction = CONFIG.get("color_correction", False)
modules.globals.opacity = CONFIG.get("opacity", 1.0)
modules.globals.sharpness = CONFIG.get("sharpness", 0.0)
modules.globals.enable_interpolation = CONFIG.get("enable_interpolation", False)
modules.globals.interpolation_weight = CONFIG.get("interpolation_weight", 0.0)
modules.globals.headless = True
modules.globals.source_path = "/content/Deep-Live-Cam/source_face.png"
modules.globals.nsfw_filter = False
modules.globals.log_level = "error"
for ek in ("face_enhancer", "face_enhancer_gpen256", "face_enhancer_gpen512"):
    modules.globals.fp_ui[ek] = ek in modules.globals.frame_processors
print(f"[✓] Globals set  |  providers={modules.globals.execution_providers}")
print(f"    processors={modules.globals.frame_processors}")

# Load source face
from modules.face_analyser import get_one_face, detect_one_face_fast, get_face_analyser
SOURCE_PATH = "/content/Deep-Live-Cam/source_face.png"
source_img = cv2.imread(SOURCE_PATH)
if source_img is None:
    raise FileNotFoundError(f"Source face not found at {SOURCE_PATH}. Run Cell 2.")
print("[*] Loading face analyser (InsightFace buffalo_l)...")
get_face_analyser()
print("[✓] Face analyser loaded")
source_face = get_one_face(source_img)
if source_face is None:
    raise ValueError("No face in source image — upload a better image in Cell 2.")
print(f"[✓] Source face loaded (score={source_face.det_score:.3f})")

# Load processors
from modules.processors.frame.core import get_frame_processors_modules
from modules.processors.frame.face_swapper import (
    pre_check as swapper_pre_check, pre_start as swapper_pre_start,
    get_face_swapper, process_frame as swapper_process_frame,
)
print("[*] Downloading face swap model (inswapper_128.onnx)...")
swapper_pre_check()
print("[✓] Model downloaded")
print("[*] Initializing ONNX session (CUDA)...")
swapper_pre_start()
swapper = get_face_swapper()
if swapper is None:
    raise RuntimeError("Failed to load face swapper!")
print("[✓] Face swapper ready (CUDA)")

processor_modules = get_frame_processors_modules(modules.globals.frame_processors)
for pm in processor_modules:
    pm.pre_check()
    print(f"[✓] Processor ready: {pm.NAME}")

# ── Performance Metrics ──────────────────────────────────────

class PerfMetrics:
    """Thread-safe performance metrics."""
    def __init__(self):
        self.lock = threading.Lock()
        self.input_fps = 0.0
        self.output_fps = 0.0
        self.inference_ms = 0.0
        self.queue_length = 0
        self.connected_clients = 0
        self.frames_processed = 0
        self.frames_dropped = 0
        from collections import deque
        self._input_t = deque(maxlen=60)
        self._output_t = deque(maxlen=60)
        self._infer_t = deque(maxlen=30)

    def record_input(self):
        now = time.time()
        with self.lock:
            self._input_t.append(now)
            if len(self._input_t) >= 2:
                dt = self._input_t[-1] - self._input_t[0]
                if dt > 0:
                    self.input_fps = (len(self._input_t) - 1) / dt

    def record_output(self):
        now = time.time()
        with self.lock:
            self._output_t.append(now)
            self.frames_processed += 1
            if len(self._output_t) >= 2:
                dt = self._output_t[-1] - self._output_t[0]
                if dt > 0:
                    self.output_fps = (len(self._output_t) - 1) / dt

    def record_inference(self, ms):
        with self.lock:
            self._infer_t.append(ms)
            self.inference_ms = sum(self._infer_t) / len(self._infer_t)

    def record_drop(self):
        with self.lock:
            self.frames_dropped += 1

    def snapshot(self):
        with self.lock:
            return dict(
                input_fps=round(self.input_fps, 1),
                output_fps=round(self.output_fps, 1),
                inference_ms=round(self.inference_ms, 1),
                queue_length=self.queue_length,
                connected_clients=self.connected_clients,
                frames_processed=self.frames_processed,
                frames_dropped=self.frames_dropped,
            )

metrics = PerfMetrics()

# ── WebRTC Server ────────────────────────────────────────

PROC_RES = CONFIG.get("processing_resolution", 640)
FPS_LIMIT = CONFIG.get("fps_limit", 30)
DET_CONF = CONFIG.get("face_detection_confidence", 0.5)

def _process_sync(bgr):
    """Run face swap processing."""
    try:
        h, w = bgr.shape[:2]
        scale = 1.0
        if max(h, w) > PROC_RES:
            scale = PROC_RES / max(h, w)
            nw, nh = int(w * scale), int(h * scale)
            proc = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
        else:
            proc = bgr

        t0 = time.time()
        target = detect_one_face_fast(proc) if not modules.globals.many_faces else None
        if target is not None and target.det_score < DET_CONF:
            target = None

        if target is not None or modules.globals.many_faces:
            for pm in processor_modules:
                try:
                    proc = pm.process_frame(source_face, proc, target_face=target)
                except TypeError:
                    proc = pm.process_frame(source_face, proc)

        metrics.record_inference((time.time() - t0) * 1000)
        if scale != 1.0:
            proc = cv2.resize(proc, (w, h), interpolation=cv2.INTER_LINEAR)
        metrics.record_output()
        return proc
    except Exception as e:
        logging.error(f"Processing error: {e}")
        return bgr

class InferenceTransformTrack(MediaStreamTrack):
    kind = "video"
    def __init__(self, track, loop):
        super().__init__()
        self.track = track
        self.loop = loop
        self.in_q = asyncio.Queue(maxsize=2)
        self.out_q = asyncio.Queue(maxsize=2)
        self.last_processed = None
        self.worker_task = asyncio.ensure_future(self._worker())

    async def _worker(self):
        while True:
            try:
                frame = await self.in_q.get()
                img = frame.to_ndarray(format="bgr24")
                processed = await self.loop.run_in_executor(None, _process_sync, img)
                try:
                    self.out_q.put_nowait(processed)
                except asyncio.QueueFull:
                    try: self.out_q.get_nowait()
                    except: pass
                    try: self.out_q.put_nowait(processed)
                    except: pass
            except asyncio.CancelledError:
                break
            except Exception as e:
                logging.error(f"Worker exception: {e}")

    async def recv(self):
        frame = await self.track.recv()
        metrics.record_input()
        try:
            self.in_q.put_nowait(frame)
        except asyncio.QueueFull:
            metrics.record_drop()
            try: self.in_q.get_nowait()
            except: pass
            try: self.in_q.put_nowait(frame)
            except: pass

        try:
            self.last_processed = self.out_q.get_nowait()
        except asyncio.QueueEmpty:
            pass

        if self.last_processed is not None:
            new_frame = VideoFrame.from_ndarray(self.last_processed, format="bgr24")
            new_frame.pts = frame.pts
            new_frame.time_base = frame.time_base
            return new_frame
        return frame

pcs = set()
RTC_CFG = RTCConfiguration(iceServers=[
    RTCIceServer(urls=["stun:stun.l.google.com:19302"]),
    RTCIceServer(urls=["turn:openrelay.metered.ca:80"], username="openrelayproject", credential="openrelayproject"),
    RTCIceServer(urls=["turn:openrelay.metered.ca:443"], username="openrelayproject", credential="openrelayproject"),
    RTCIceServer(urls=["turn:openrelay.metered.ca:443?transport=tcp"], username="openrelayproject", credential="openrelayproject"),
])

async def handle_offer(request):
    params = await request.json()
    offer = RTCSessionDescription(sdp=params["sdp"], type=params["type"])
    pc = RTCPeerConnection(configuration=RTC_CFG)
    pcs.add(pc)
    metrics.connected_clients = len(pcs)
    print(f"[WebRTC] New peer (total: {len(pcs)})")

    @pc.on("connectionstatechange")
    async def on_state():
        st = pc.connectionState
        print(f"[WebRTC] State: {st}")
        if st in ("failed", "closed", "disconnected"):
            await pc.close()
            pcs.discard(pc)
            metrics.connected_clients = len(pcs)
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except ImportError:
                pass

    @pc.on("track")
    def on_track(track):
        print(f"[WebRTC] Track: {track.kind}")
        if track.kind == "video":
            loop = asyncio.get_event_loop()
            local_video = InferenceTransformTrack(track, loop)
            pc.addTrack(local_video)

    await pc.setRemoteDescription(offer)
    answer = await pc.createAnswer()
    await pc.setLocalDescription(answer)
    return web.json_response({"sdp": pc.localDescription.sdp, "type": pc.localDescription.type})

async def handle_health(request):
    snap = metrics.snapshot()
    return web.json_response({"status": "ok", "metrics": snap})

async def handle_index(request):
    return web.Response(text="<h1>Deep-Live-Cam WebSocket Server</h1>", content_type="text/html")

app = web.Application()
app.router.add_get("/", handle_index)
app.router.add_post("/offer", handle_offer)
app.router.add_get("/health", handle_health)

def _run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    runner = web.AppRunner(app)
    loop.run_until_complete(runner.setup())
    site = web.TCPSite(runner, "0.0.0.0", SERVER_PORT)
    loop.run_until_complete(site.start())
    print(f"\n[✓] Server on port {SERVER_PORT}")
    print(f"[✓] Ngrok: {_ngrok_url}")
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  🎭 READY — run client.py with:")
    print(f"  python client.py --url {_ngrok_url}")
    print(f"{sep}\n")
    loop.run_forever()

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("[✓] Server running. Proceed to Cell 7.")

## 7️⃣ Generate Local Client
Downloads `client.py` for your local PC. Run it to start streaming.

In [ ]:
#@title Generate client.py
#@markdown Downloads the local Python client for your PC.

CLIENT_CODE = r'''
#!/usr/bin/env python3
# You only need: aiohttp, opencv-python, numpy, pyvirtualcam (optional)

import argparse
import asyncio
import logging
import time
import aiohttp
import cv2
import numpy as np

logger = logging.getLogger("dlc-client")

async def run_client(url, camera, w, h, fps, use_vcam, vw, vh, retries, delay):
    for attempt in range(1, retries + 1):
        if attempt > 1:
            logger.info(f"Retry {attempt}/{retries} in {delay}s...")
            await asyncio.sleep(delay)
        try:
            await _stream(url, camera, w, h, fps, use_vcam, vw, vh)
            return
        except KeyboardInterrupt:
            return
        except Exception as e:
            logger.error(f"Connection error: {e}")
    logger.error(f"Max retries ({retries}) exceeded.")

async def _stream(url, camera, w, h, fps, use_vcam, vw, vh):
    ws_url = url.rstrip("/").replace("https://", "wss://").replace("http://", "ws://") + "/ws"
    cap = cv2.VideoCapture(camera)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)
    cap.set(cv2.CAP_PROP_FPS, fps)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open camera {camera}")
    aw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    ah = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    logger.info(f"Camera opened: {aw}x{ah}")
    vcam_dev = None
    if use_vcam:
        try:
            import pyvirtualcam
            vcam_dev = pyvirtualcam.Camera(width=vw, height=vh, fps=fps, fmt=pyvirtualcam.PixelFormat.BGR)
            logger.info(f"Virtual cam: {vcam_dev.device}")
        except Exception as e:
            logger.warning(f"VCam unavailable: {e}")
    logger.info(f"Connecting to {ws_url} ...")
    async with aiohttp.ClientSession() as session:
        hdrs = {"ngrok-skip-browser-warning": "true"}
        async with session.ws_connect(ws_url, headers=hdrs, max_msg_size=10*1024*1024) as ws:
            logger.info("Connected!")
            print("\n" + "=" * 50)
            print("  Deep-Live-Cam Client Running (WebSocket)")
            print("  'q' = quit  |  'm' = mirror")
            print("=" * 50 + "\n")
            mirror = False
            frames_count = 0
            try:
                while True:
                    for _ in range(3): cap.grab()
                    ret, frame = cap.read()
                    if not ret:
                        logger.error("Camera read failed")
                        break
                    _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 60])
                    t0 = time.time()
                    await ws.send_bytes(buf.tobytes())
                    msg = await ws.receive()
                    if msg.type == aiohttp.WSMsgType.BINARY:
                        dt = time.time() - t0
                        fps_val = 1.0 / dt if dt > 0 else 0
                        frames_count += 1
                        arr = np.frombuffer(msg.data, np.uint8)
                        out_frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
                        disp = cv2.flip(out_frame, 1) if mirror else out_frame.copy()
                        cv2.putText(disp, f"Ping: {dt*1000:.0f}ms | FPS: {fps_val:.1f} | #{frames_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                        cv2.imshow("Deep-Live-Cam", disp)
                        if vcam_dev:
                            try:
                                vcam_dev.send(cv2.resize(out_frame, (vw, vh)))
                                vcam_dev.sleep_until_next_frame()
                            except Exception: pass
                    elif msg.type in (aiohttp.WSMsgType.CLOSED, aiohttp.WSMsgType.ERROR):
                        logger.error("WebSocket closed by server.")
                        break
                    key = cv2.waitKey(1) & 0xFF
                    if key == ord("q"): break
                    elif key == ord("m"): mirror = not mirror
            finally:
                cv2.destroyAllWindows()
                cap.release()
                if vcam_dev: vcam_dev.close()

def main():
    ap = argparse.ArgumentParser(description="Deep-Live-Cam WebSocket Client")
    ap.add_argument("--url", required=True, help="Colab ngrok URL")
    ap.add_argument("--camera", type=int, default=0, help="Camera index")
    ap.add_argument("--width", type=int, default=640, help="Capture width")
    ap.add_argument("--height", type=int, default=480, help="Capture height")
    ap.add_argument("--fps", type=int, default=30, help="Capture FPS")
    ap.add_argument("--vcam", action="store_true", help="Enable virtual camera")
    ap.add_argument("--vcam-width", type=int, default=1280, help="VCam width")
    ap.add_argument("--vcam-height", type=int, default=720, help="VCam height")
    ap.add_argument("--retries", type=int, default=10, help="Max retries")
    ap.add_argument("--retry-delay", type=float, default=3.0, help="Retry delay")
    ap.add_argument("--verbose", action="store_true", help="Enable debug")
    a = ap.parse_args()
    try:
        asyncio.run(run_client(a.url, a.camera, a.width, a.height, a.fps,
                               a.vcam, a.vcam_width, a.vcam_height,
                               a.retries, a.retry_delay))
    except KeyboardInterrupt:
        print("\nShutting down...")


if __name__ == "__main__":
    main()

'''

# Write client.py
for p in ["/content/client.py", "/content/Deep-Live-Cam/client.py"]:
    with open(p, "w") as f:
        f.write(CLIENT_CODE)
print("[✓] client.py generated")

# Download
from google.colab import files
files.download("/content/client.py")

print()
print("=" * 60)
print("  📥 CLIENT INSTRUCTIONS")
print("=" * 60)
print()
print("  1. Install deps on LOCAL PC:")
print("     pip install aiortc aiohttp opencv-python av pyvirtualcam")
print()
print(f"  2. Run:  python client.py --url {_ngrok_url}")
print(f"  3. VCam: python client.py --url {_ngrok_url} --vcam")
print()
print("  Keys: q=quit, m=mirror")
print("  Flags: --verbose, --camera 1, --width 320 --height 240")
print("=" * 60)

## 8️⃣ Monitoring Dashboard
Run this cell to see live GPU / performance metrics. Re-run to refresh.

In [ ]:
#@title Monitoring Dashboard (auto-refreshes every 3s)
#@markdown Displays GPU utilization, FPS, latency, and connection info.

import subprocess, time
from IPython.display import display, HTML, clear_output

def gpu_stats():
    try:
        r = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5)
        if r.returncode == 0:
            p = r.stdout.strip().split(", ")
            return dict(util=f"{p[0]}%", used=f"{p[1]} MB",
                        total=f"{p[2]} MB", temp=f"{p[3]}°C")
    except Exception:
        pass
    return dict(util="N/A", used="N/A", total="N/A", temp="N/A")

def render():
    s = metrics.snapshot()
    g = gpu_stats()
    return f"""
    <div style="font-family:monospace;background:#0d1117;color:#c9d1d9;
                padding:20px;border-radius:10px;border:1px solid #30363d">
      <h2 style="color:#58a6ff;margin-top:0">🎭 Deep-Live-Cam Monitor</h2>
      <p style="color:#8b949e">Updated: {time.strftime("%H:%M:%S")}</p>
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">
        <div style="background:#161b22;padding:12px;border-radius:8px;border:1px solid #30363d">
          <h3 style="color:#7ee787;margin-top:0">🖥️ GPU</h3>
          <table style="width:100%;color:#c9d1d9">
            <tr><td>Utilization</td><td style="text-align:right;color:#58a6ff">{g["util"]}</td></tr>
            <tr><td>VRAM</td><td style="text-align:right;color:#58a6ff">{g["used"]} / {g["total"]}</td></tr>
            <tr><td>Temperature</td><td style="text-align:right;color:#58a6ff">{g["temp"]}</td></tr>
          </table>
        </div>
        <div style="background:#161b22;padding:12px;border-radius:8px;border:1px solid #30363d">
          <h3 style="color:#7ee787;margin-top:0">📊 Performance</h3>
          <table style="width:100%;color:#c9d1d9">
            <tr><td>Input FPS</td><td style="text-align:right;color:#58a6ff">{s["input_fps"]}</td></tr>
            <tr><td>Output FPS</td><td style="text-align:right;color:#58a6ff">{s["output_fps"]}</td></tr>
            <tr><td>Inference</td><td style="text-align:right;color:#58a6ff">{s["inference_ms"]} ms</td></tr>
          </table>
        </div>
        <div style="background:#161b22;padding:12px;border-radius:8px;border:1px solid #30363d">
          <h3 style="color:#7ee787;margin-top:0">🔗 Connections</h3>
          <table style="width:100%;color:#c9d1d9">
            <tr><td>Clients</td><td style="text-align:right;color:#58a6ff">{s["connected_clients"]}</td></tr>
            <tr><td>Processed</td><td style="text-align:right;color:#58a6ff">{s["frames_processed"]}</td></tr>
            <tr><td>Dropped</td><td style="text-align:right;color:#f85149">{s["frames_dropped"]}</td></tr>
          </table>
        </div>
        <div style="background:#161b22;padding:12px;border-radius:8px;border:1px solid #30363d">
          <h3 style="color:#7ee787;margin-top:0">⚙️ Config</h3>
          <table style="width:100%;color:#c9d1d9">
            <tr><td>Processors</td><td style="text-align:right;color:#58a6ff">{", ".join(modules.globals.frame_processors)}</td></tr>
            <tr><td>Resolution</td><td style="text-align:right;color:#58a6ff">{PROC_RES}px</td></tr>
            <tr><td>FPS Limit</td><td style="text-align:right;color:#58a6ff">{FPS_LIMIT}</td></tr>
          </table>
        </div>
      </div>
      <p style="color:#8b949e;margin-bottom:0;margin-top:12px">Ngrok: <code style="color:#58a6ff">{_ngrok_url}</code></p>
    </div>
    """

REFRESH = 30  # iterations (30 × 3s = 90s, re-run to continue)
for _ in range(REFRESH):
    clear_output(wait=True)
    display(HTML(render()))
    time.sleep(3)
print("Dashboard paused — re-run cell to continue.")

## 🔧 Troubleshooting

### Common Issues

| Issue | Solution |
|-------|----------|
| "No CUDA GPU found" | Runtime → Change runtime type → GPU |
| "Failed to read from camera" | `--camera 1` or `--camera 2` |
| "Connection refused" | Ensure Cell 5 is running and ngrok URL is correct |
| "No face detected" in source | Upload a clearer front-facing photo |
| High latency | Reduce `processing_resolution` to 320 or 480 |
| Choppy output | Reduce `fps_limit` to 15-20 |
| OOM error | Reduce `max_memory_gb`; use only `face_swapper` |
| Ngrok drops | Auto-reconnect monitor handles this |
| Black/frozen video | Restart Cell 5, reconnect client |

### Performance Tips
- **Lowest latency**: Use only `face_swapper` (~30-50ms/frame on T4)
- **Faster inference**: Set `processing_resolution` to 480
- **Less bandwidth**: Use `--width 320 --height 240` on client
- **Disable extras**: Turn off `mouth_mask`, `poisson_blend`, `color_correction`

### Client Installation
```bash
pip install aiortc aiohttp opencv-python av pyvirtualcam
python client.py --url https://xxxx.ngrok-free.app
```